# Unsupervised Anomaly Detection in LHC Event Data

**Saqib Umair**  
MSc Particle and Nuclear Physics, University of Edinburgh

This notebook summarises the public, reproducible version of the analysis. The autoencoder is trained only on Standard Model (SM) background events. Mean-squared reconstruction error is then used as an anomaly score for nine Beyond Standard Model (BSM) benchmark samples.

The analysis uses the channel-2b setup from the Dark Machines Anomaly Score Challenge. Raw challenge data are not redistributed here.

## 1. Event representation

Each event is converted into a fixed 39-feature vector:

- five object multiplicities: electrons, muons, jets, b-jets and photons;
- `log(MET)` and `METphi`;
- the eight highest-energy reconstructed objects, each described by `log(E)`, `log(pT)`, `eta` and `phi`.

The SM sample contains 340,268 events. An 80/20 split gives 272,214 training events and 68,054 test events. The MinMax scaler is fitted on the SM training split only.

In [ ]:
"""Preprocessing utilities for the Dark Machines channel-2b event format."""

from __future__ import annotations

import os
from pathlib import Path

import numpy as np


N_PARTICLES = 8
N_PARTICLE_FEATURES = 4
LOG_EPSILON = 1e-6


def find_data_start_line(filepath: str | os.PathLike[str]) -> int:
    """Return the number of non-data lines before the first event."""
    filepath = str(filepath)
    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)

    with open(filepath, "r", encoding="utf-8") as handle:
        for i, line in enumerate(handle):
            line = line.strip()
            if not line:
                continue
            try:
                parts = line.split(";")
                if len(parts) >= 5:
                    float(parts[3])
                    float(parts[4])
                    return i
            except ValueError:
                continue

    raise ValueError(f"No valid event row found in {filepath}")


def process_file(filepath: str | os.PathLike[str], skip_rows: int = 0) -> np.ndarray:
    """Convert raw events into the 39-feature representation used by the model."""
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(filepath)

    event_features: list[list[float]] = []

    with filepath.open("r", encoding="utf-8") as handle:
        for _ in range(skip_rows):
            next(handle)

        for line in handle:
            line = line.strip()
            if not line:
                continue

            parts = line.split(";")
            met = float(parts[3])
            met_phi = float(parts[4])
            particle_objects = parts[5:]

            particles: list[tuple[float, float, float, float, str]] = []
            counts = {"ele": 0, "muon": 0, "jet": 0, "bjet": 0, "photon": 0}

            for object_string in particle_objects:
                if not object_string:
                    continue
                values = object_string.split(",")
                if len(values) < 5:
                    continue

                object_type = values[0]
                if object_type in ("e+", "e-"):
                    counts["ele"] += 1
                elif object_type in ("m+", "m-"):
                    counts["muon"] += 1
                elif object_type == "j":
                    counts["jet"] += 1
                elif object_type == "b":
                    counts["bjet"] += 1
                elif object_type == "g":
                    counts["photon"] += 1

                particles.append(
                    (
                        float(values[1]),
                        float(values[2]),
                        float(values[3]),
                        float(values[4]),
                        object_type,
                    )
                )

            particles.sort(key=lambda particle: particle[0], reverse=True)

            features = [
                counts["ele"],
                counts["muon"],
                counts["jet"],
                counts["bjet"],
                counts["photon"],
                np.log(met + LOG_EPSILON),
                met_phi,
            ]

            for i in range(N_PARTICLES):
                if i < len(particles):
                    energy, pt, eta, phi, _ = particles[i]
                    features.extend(
                        [
                            np.log(energy + LOG_EPSILON),
                            np.log(pt + LOG_EPSILON),
                            eta,
                            phi,
                        ]
                    )
                else:
                    features.extend([0.0] * N_PARTICLE_FEATURES)

            event_features.append(features)

    return np.asarray(event_features, dtype=np.float32)


## 2. Autoencoder

The network is deliberately compact:

```text
39 -> 32 -> 8 -> 32 -> 39
```

The 8-dimensional bottleneck forces compression of the SM event representation. The output layer uses a sigmoid because the engineered features are scaled to [0, 1].

In [ ]:
"""Fully connected autoencoder used for the LHC anomaly-detection study."""

import torch
from torch import nn


class AutoEncoder(nn.Module):
    """39 -> 32 -> 8 -> 32 -> 39 autoencoder."""

    def __init__(self, input_dim: int = 39, latent_dim: int = 8) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


## 3. Training

The model is trained for 30 epochs with Adam, a learning rate of 1e-3 and mean-squared reconstruction loss. Validation loss is evaluated at every epoch and the checkpoint with the lowest validation loss is retained.

This is validation-based checkpointing rather than early termination: the training loop runs the full 30 epochs.

The best validation loss in the saved run was approximately **0.004652**.

In [ ]:
"""Training loop used for the compact SM-background autoencoder."""

from __future__ import annotations

from pathlib import Path

import torch
from torch import nn, optim


def train_model(
    train_loader,
    validation_loader,
    model: nn.Module,
    epochs: int = 30,
    learning_rate: float = 1e-3,
    save_path: str = "best_autoencoder.pt",
):
    """Train the model and save the checkpoint with the lowest validation loss."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_validation_loss = float("inf")
    history = {"train_loss": [], "validation_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for data, _ in train_loader:
            reconstruction = model(data)
            loss = criterion(reconstruction, data)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * data.size(0)

        train_loss /= len(train_loader.dataset)
        history["train_loss"].append(train_loss)

        model.eval()
        validation_loss = 0.0
        with torch.no_grad():
            for data, _ in validation_loader:
                reconstruction = model(data)
                validation_loss += criterion(reconstruction, data).item() * data.size(0)

        validation_loss /= len(validation_loader.dataset)
        history["validation_loss"].append(validation_loss)

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            torch.save(model.state_dict(), save_path)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train={train_loss:.6f} | validation={validation_loss:.6f}"
        )

    return history, best_validation_loss


## 4. Anomaly score and ROC evaluation

For each event, the anomaly score is the mean squared reconstruction error across the 39 input features. The same score is evaluated for held-out SM events and all BSM benchmarks.

In [ ]:
"""Evaluation helpers for reconstruction-error anomaly detection."""

from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, roc_curve
from torch import nn


def anomaly_scores(model: nn.Module, data: np.ndarray, batch_size: int = 256) -> np.ndarray:
    """Return mean-squared reconstruction error for each event."""
    model.eval()
    tensor = torch.as_tensor(data, dtype=torch.float32)
    scores: list[np.ndarray] = []

    with torch.no_grad():
        for start in range(0, len(tensor), batch_size):
            batch = tensor[start : start + batch_size]
            reconstruction = model(batch)
            score = torch.mean((reconstruction - batch) ** 2, dim=1)
            scores.append(score.cpu().numpy())

    return np.concatenate(scores)


def roc_metrics(sm_scores: np.ndarray, bsm_scores: np.ndarray):
    """Return false-positive rate, true-positive rate and ROC AUC."""
    labels = np.concatenate(
        [np.zeros(len(sm_scores), dtype=int), np.ones(len(bsm_scores), dtype=int)]
    )
    scores = np.concatenate([sm_scores, bsm_scores])
    fpr, tpr, thresholds = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    return fpr, tpr, thresholds, auc


## 5. Results

| BSM benchmark | ROC AUC |
|---|---:|
| gluino_1000.0_neutralino_1.0 | **0.982** |
| stlp_st1000 | **0.957** |
| chacha_cha600_neut200 | 0.679 |
| chacha_cha400_neut60 | 0.636 |
| pp24mt_50 | 0.621 |
| chaneut_cha200_neut50 | 0.578 |
| chaneut_cha250_neut150 | 0.570 |
| chacha_cha300_neut140 | 0.557 |
| pp23mt_50 | **0.490** |

The strongest separation is obtained for the gluino and stau-like benchmarks. Several other signals overlap substantially with the SM background, while `pp23mt_50` is effectively indistinguishable from background with this score.

The variation in AUC is a useful result in itself: a background-trained autoencoder does not provide uniform sensitivity to every BSM topology.